# ParlaMint (sample) dataset processing

Input data: ParlaMint dataset (handle), where the data is merged together into a master preprocessed dataset, to be used for all of the analysis. Specifically, we include the following data:
- ```Sample/ParlaMint-SI_5.1.txt``` (text files, to acquire the texts)
- ```Sample/ParlaMint-SI_5.1.connllu``` (metadata and sentiment metadata files)

Output:
- Pre-processed and merged master dataset [ParlaMint-SI_5.1_master.tsv](../Datasets/)

In [1]:
import pandas as pd
import os
import glob

In [2]:
text_dir = "../Sample/ParlaMint-SI.txt"
conllu_dir = "../Sample/ParlaMint-SI.conllu"


In [3]:
# Text files to extrac ID and text
text_files = glob.glob(os.path.join(text_dir, '*', '*.txt'))

#Session metadata
text_meta = glob.glob(os.path.join(text_dir, '*', '*.tsv'))

#Sentiment-related metadata
sent_files = glob.glob(os.path.join(conllu_dir, '**', '*-ana-meta-en.tsv'))

print(text_files)

['../Sample/ParlaMint-SI.txt/2014/ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.txt', '../Sample/ParlaMint-SI.txt/2022/ParlaMint-SI_2022-01-13-SDZ8-Izredna-93.txt', '../Sample/ParlaMint-SI.txt/2007/ParlaMint-SI_2007-01-29-SDZ4-Redna-24.txt', '../Sample/ParlaMint-SI.txt/2000/ParlaMint-SI_2000-10-27-SDZ3-Redna-01.txt']


In [4]:
id = []
texts = []
#Processing text files
for file in text_files:
    with open(file, 'r', encoding="utf-8") as infile:
        for line in infile:
            if '\t' in line:
                parts = line.strip().split('\t')
                if len(parts) == 2:
                    id.append(parts[0])
                    texts.append(parts[1])


df_texts = pd.DataFrame({'ID': id, 'Text':texts})
print("df_meta shape: ", df_texts.shape)

df_texts.head()


df_meta shape:  (420, 2)


,ID,Text
0,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u1,Spoštovani kolegice poslanke in kolegi poslanc...
1,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u2,"Spoštovani predsednik, spoštovani kolegice in ..."
2,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u3,Hvala za dopolnilno obrazložitev. Predlog prip...
3,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u4,"Hvala lepa za besedo, gospod predsednik. Spošt..."
4,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u5,Hvala tudi vam za podano poročilo. Gospod Samo...


In [5]:
df_meta = pd.DataFrame()
for file in text_meta:
    df = pd.read_csv(file, sep='\t', encoding='utf-8')
    df_meta = pd.concat([df_meta, df])

df_meta.head()

cols = ['Text_ID', 'Title', 'Body', 'Session', 'Sitting', 'Agenda', 'Lang']
df_meta = df_meta.drop(columns=cols)

print("df_meta shape: ", df_meta.shape)
df_meta.head()


df_meta shape:  (420, 17)


,ID,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,Speaker_party_name,Party_status,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth,Topic
0,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u1,2014-01-24,6. mandat,Izredna,Reference,Chairperson,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,VeberJanko,"Veber, Janko",M,1960,Other
1,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u2,2014-01-24,6. mandat,Izredna,Reference,Regular,MP,notMinister,SDS,Slovenian Democratic Party,Opposition,Right,VizjakAndrej,"Vizjak, Andrej",M,1964,Energy
2,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u3,2014-01-24,6. mandat,Izredna,Reference,Chairperson,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,VeberJanko,"Veber, Janko",M,1960,Housing
3,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u4,2014-01-24,6. mandat,Izredna,Reference,Regular,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,BevkSamo,"Bevk, Samo",M,1956,Energy
4,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u5,2014-01-24,6. mandat,Izredna,Reference,Chairperson,MP,notMinister,SD,Social Democrats,Coalition,Centre-left,VeberJanko,"Veber, Janko",M,1960,Housing


In [6]:
df_sent = pd.DataFrame()
for file in sent_files:
    df = pd.read_csv(file, sep='\t', encoding='utf-8')
    df_sent = pd.concat([df_sent, df])

df_sent = df_sent[df_sent['Element'] == 'u']
cols = ['ID', 'Senti_3', 'Senti_6', 'Senti_n', 'Sents', 'Words', 'Tokens']
df_sent = df_sent[cols].reset_index(drop=True)
df_sent['ID'] = df_sent['ID'].str.replace(".ana", "", regex=False)

print("df_sent shape: ", df_sent.shape)
df_sent.head()


df_sent shape:  (420, 7)


,ID,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens
0,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u1,Neutral,neutral positive,3.40,24,311,363
1,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u2,Negative,mixed negative,0.80,68,1638,1889
2,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u3,Neutral,neutral positive,2.97,3,28,31
3,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u4,Neutral,neutral negative,2.08,31,711,785
4,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u5,Neutral,neutral positive,3.00,2,20,24


In [9]:
df = pd.merge(pd.merge(df_texts, df_meta, on='ID'), df_sent, on='ID')

print("df shape: ", df.shape)
df.head()

df shape:  (420, 24)


,ID,Text,Date,Term,Meeting,Subcorpus,Speaker_role,Speaker_MP,Speaker_minister,Speaker_party,...,Speaker_name,Speaker_gender,Speaker_birth,Topic,Senti_3,Senti_6,Senti_n,Sents,Words,Tokens
0,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u1,Spoštovani kolegice poslanke in kolegi poslanc...,2014-01-24,6. mandat,Izredna,Reference,Chairperson,MP,notMinister,SD,...,"Veber, Janko",M,1960,Other,Neutral,neutral positive,3.40,24,311,363
1,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u2,"Spoštovani predsednik, spoštovani kolegice in ...",2014-01-24,6. mandat,Izredna,Reference,Regular,MP,notMinister,SDS,...,"Vizjak, Andrej",M,1964,Energy,Negative,mixed negative,0.80,68,1638,1889
2,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u3,Hvala za dopolnilno obrazložitev. Predlog prip...,2014-01-24,6. mandat,Izredna,Reference,Chairperson,MP,notMinister,SD,...,"Veber, Janko",M,1960,Housing,Neutral,neutral positive,2.97,3,28,31
3,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u4,"Hvala lepa za besedo, gospod predsednik. Spošt...",2014-01-24,6. mandat,Izredna,Reference,Regular,MP,notMinister,SD,...,"Bevk, Samo",M,1956,Energy,Neutral,neutral negative,2.08,31,711,785
4,ParlaMint-SI_2014-01-24-SDZ6-Izredna-53.u5,Hvala tudi vam za podano poročilo. Gospod Samo...,2014-01-24,6. mandat,Izredna,Reference,Chairperson,MP,notMinister,SD,...,"Veber, Janko",M,1960,Housing,Neutral,neutral positive,3.00,2,20,24


In [10]:
#Dataset saved as CSV/TSV
df.to_csv("../Datasets/ParlaMint-SI_5.1_master.tsv", sep='\t', encoding='utf-8')